# PBMC 10k: Rust vs Python Inference Speed

Benchmarks `fit_all_grid_points` on `processed_pbmc_10k_raw.h5ad` (10,997 cells × 36,601 genes)
comparing the Rust PSS fast-path against the pure-Python baseline.

**Model:** Bursty + Poisson  
**Parallelism levels tested:**
- `num_cores=1` — sequential scipy per gene  
- `num_cores=N` — ThreadPoolExecutor with N scipy workers (Rust PSS fires inside each thread)  

Each configuration is run with `_HAS_RUST=True` and `_HAS_RUST=False` to isolate the
contribution of the Rust PSS fast-path from thread-level parallelism.

## Setup

In [ ]:
import sys, os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, 'src/monod')
warnings.filterwarnings('ignore')

import cme_toolbox
import inference
from cme_toolbox import CMEModel, _HAS_RUST
from extract_data import extract_data
from inference import InferenceParameters, searchdata_from_adata

print(f'Rust extension available: {_HAS_RUST}')
print(f'Logical CPUs: {os.cpu_count()}')

## Load and extract data

Run `extract_data` once so timing comparisons cover only inference.

In [ ]:
H5AD_PATH = 'example_h5ad/processed_pbmc_10k_raw.h5ad'
N_GENES    = 100
MODEL      = CMEModel('Bursty', 'Poisson')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    adata = extract_data(
        H5AD_PATH,
        MODEL,
        dataset_name='pbmc_speed',
        modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
        n_genes=N_GENES,
        hist_type='unique',
        viz=False,
    )

sd = searchdata_from_adata(adata)
print(f'Genes extracted: {sd.n_genes}')
print(f'M range — unspliced: {adata.uns["M"][0].min():.0f}–{adata.uns["M"][0].max():.0f},'
      f'  spliced: {adata.uns["M"][1].min():.0f}–{adata.uns["M"][1].max():.0f}')

## Benchmark helper

In [ ]:
GRADIENT_PARAMS_BASE = {
    'max_iterations': 25,
    'init_pattern': 'moments',
    'num_restarts': 1,
}

def run_inference(num_cores, has_rust):
    """Time fit_all_grid_points with given num_cores and _HAS_RUST setting."""
    gp = dict(GRADIENT_PARAMS_BASE, num_gene_cores=num_cores)
    ip = InferenceParameters(
        'pbmc_speed',
        MODEL,
        use_lengths=False,
        gradient_params=gp,
        save=False,
    )
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        # Patch _HAS_RUST in both modules for the duration of this call.
        cme_toolbox._HAS_RUST = has_rust
        inference._HAS_RUST   = has_rust
        t0 = time.perf_counter()
        ip.fit_all_grid_points(sd, num_cores=num_cores, save=False)
        elapsed = time.perf_counter() - t0
        cme_toolbox._HAS_RUST = _HAS_RUST   # restore
        inference._HAS_RUST   = _HAS_RUST
    return elapsed

## Run benchmarks

Each configuration is timed once. For a more stable estimate increase `N_RUNS`.

In [ ]:
CORE_COUNTS = [1, 2, 4, 8]

results = {}  # (num_cores, has_rust) -> elapsed seconds

np.random.seed(0)
for num_cores in CORE_COUNTS:
    for has_rust in (True, False):
        label = f'cores={num_cores}, rust={has_rust}'
        print(f'Running {label} ...', end=' ', flush=True)
        t = run_inference(num_cores, has_rust)
        results[(num_cores, has_rust)] = t
        print(f'{t:.1f}s')

## Results

In [ ]:
print(f'{"cores":>6}  {"Rust (s)":>10}  {"Python (s)":>10}  {"Speedup":>8}')
print('-' * 40)
for nc in CORE_COUNTS:
    t_rust = results[(nc, True)]
    t_py   = results[(nc, False)]
    print(f'{nc:>6}  {t_rust:>10.1f}  {t_py:>10.1f}  {t_py/t_rust:>7.2f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

rust_times   = [results[(nc, True)]  for nc in CORE_COUNTS]
python_times = [results[(nc, False)] for nc in CORE_COUNTS]
speedups     = [results[(nc, False)] / results[(nc, True)] for nc in CORE_COUNTS]

x = np.arange(len(CORE_COUNTS))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, rust_times,   w, label='Rust',   color='steelblue')
ax.bar(x + w/2, python_times, w, label='Python', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'{nc}' for nc in CORE_COUNTS])
ax.set_xlabel('num_cores')
ax.set_ylabel('Wall time (s)')
ax.set_title(f'Inference time — {N_GENES} genes, Bursty+Poisson')
ax.legend()

ax = axes[1]
ax.plot(CORE_COUNTS, speedups, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray', linewidth=1)
ax.set_xlabel('num_cores')
ax.set_ylabel('Rust speedup over Python')
ax.set_title('Rust speedup')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.tight_layout()
plt.show()

## Gene-count sweep

Fixed `num_cores=4`; vary the number of genes fitted.
Shows how parallelism and Rust speedup scale with gene count.

In [ ]:
GENE_COUNTS  = [100, 250, 500, 1000]
SWEEP_CORES  = 4

gene_results = {}  # (n_genes, has_rust) -> elapsed

np.random.seed(0)
for n_genes in GENE_COUNTS:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        adata_g = extract_data(
            H5AD_PATH, MODEL,
            dataset_name='pbmc_gene_sweep',
            modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
            n_genes=n_genes,
            hist_type='unique',
            viz=False,
        )
    sd_g = searchdata_from_adata(adata_g)
    gp   = dict(GRADIENT_PARAMS_BASE, num_gene_cores=SWEEP_CORES)
    for has_rust in (True, False):
        label = f'n_genes={n_genes}, rust={has_rust}'
        print(f'Running {label} ...', end=' ', flush=True)
        ip = InferenceParameters('pbmc_gene_sweep', MODEL,
                                 use_lengths=False, gradient_params=gp, save=False)
        cme_toolbox._HAS_RUST = has_rust
        inference._HAS_RUST   = has_rust
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            t0 = time.perf_counter()
            ip.fit_all_grid_points(sd_g, num_cores=SWEEP_CORES, save=False)
            t  = time.perf_counter() - t0
        cme_toolbox._HAS_RUST = _HAS_RUST
        inference._HAS_RUST   = _HAS_RUST
        gene_results[(n_genes, has_rust)] = t
        print(f'{t:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

gr_rust   = [gene_results[(n, True)]  for n in GENE_COUNTS]
gr_python = [gene_results[(n, False)] for n in GENE_COUNTS]
gr_speedup = [gene_results[(n, False)] / gene_results[(n, True)] for n in GENE_COUNTS]

ax = axes[0]
ax.plot(GENE_COUNTS, gr_rust,   'o-', color='steelblue', label='Rust')
ax.plot(GENE_COUNTS, gr_python, 's-', color='coral',     label='Python')
ax.set_xlabel('Number of genes')
ax.set_ylabel('Wall time (s)')
ax.set_title(f'Inference time vs gene count (num_cores={SWEEP_CORES})')
ax.legend()

ax = axes[1]
ax.plot(GENE_COUNTS, gr_speedup, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray', linewidth=1)
ax.set_xlabel('Number of genes')
ax.set_ylabel('Rust speedup over Python')
ax.set_title('Rust speedup vs gene count')

fig.tight_layout()
plt.show()

## Cell-count sweep

Fixed `num_cores=4`, `n_genes=100`; subsample cells to vary M values.
Shows how dataset size (and thus histogram grid dimensions) affects the Rust speedup.

In [ ]:
import tempfile
import anndata as ad

CELL_COUNTS  = [100, 500, 1000, 10000]
SWEEP_GENES  = 100

full_adata   = ad.read_h5ad(H5AD_PATH)
cell_results = {}  # (n_cells, has_rust) -> (elapsed, median_grid)

rng = np.random.default_rng(0)
for n_cells in CELL_COUNTS:
    idx = rng.choice(full_adata.n_obs, size=min(n_cells, full_adata.n_obs), replace=False)
    sub = full_adata[idx].copy()
    with tempfile.NamedTemporaryFile(suffix='.h5ad', delete=False) as f:
        tmp_path = f.name
    sub.write_h5ad(tmp_path)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        adata_c = extract_data(
            tmp_path, MODEL,
            dataset_name='pbmc_cell_sweep',
            modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
            n_genes=SWEEP_GENES,
            hist_type='unique',
            viz=False,
        )
    os.unlink(tmp_path)

    sd_c = searchdata_from_adata(adata_c)
    M    = adata_c.uns['M']
    median_grid = int(np.median(M[0] * M[1]))

    gp = dict(GRADIENT_PARAMS_BASE, num_gene_cores=SWEEP_CORES)
    for has_rust in (True, False):
        label = f'n_cells={n_cells}, rust={has_rust}'
        print(f'Running {label} (median grid={median_grid}) ...', end=' ', flush=True)
        ip = InferenceParameters('pbmc_cell_sweep', MODEL,
                                 use_lengths=False, gradient_params=gp, save=False)
        cme_toolbox._HAS_RUST = has_rust
        inference._HAS_RUST   = has_rust
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            t0 = time.perf_counter()
            ip.fit_all_grid_points(sd_c, num_cores=SWEEP_CORES, save=False)
            t  = time.perf_counter() - t0
        cme_toolbox._HAS_RUST = _HAS_RUST
        inference._HAS_RUST   = _HAS_RUST
        cell_results[(n_cells, has_rust)] = (t, median_grid)
        print(f'{t:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cr_rust    = [cell_results[(n, True)][0]  for n in CELL_COUNTS]
cr_python  = [cell_results[(n, False)][0] for n in CELL_COUNTS]
cr_speedup = [cell_results[(n, False)][0] / cell_results[(n, True)][0] for n in CELL_COUNTS]
cr_grids   = [cell_results[(n, True)][1]  for n in CELL_COUNTS]

ax = axes[0]
ax.plot(CELL_COUNTS, cr_rust,   'o-', color='steelblue', label='Rust')
ax.plot(CELL_COUNTS, cr_python, 's-', color='coral',     label='Python')
ax.set_xscale('log')
ax.set_xlabel('Number of cells')
ax.set_ylabel('Wall time (s)')
ax.set_title(f'Inference time vs cell count (num_cores={SWEEP_CORES}, {SWEEP_GENES} genes)')
ax.legend()

ax = axes[1]
ax2 = ax.twiny()
ax.plot(CELL_COUNTS, cr_speedup, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Number of cells')
ax.set_ylabel('Rust speedup over Python')
ax.set_title('Rust speedup vs cell count')
ax2.set_xlim(ax.get_xlim())
ax2.set_xscale('log')
ax2.set_xticks(CELL_COUNTS)
ax2.set_xticklabels([f'M~{g}' for g in cr_grids], fontsize=8)

fig.tight_layout()
plt.show()